# Discrepancy explorer ($m$ + $m^2$)

Interactive view of FSS collapse under three discrepancy models.

Likelihood uses both $\Phi_m=m\,L^{\beta/\nu}$ and $\Phi_{m^2}=m^2\,L^{2\beta/\nu}$ (shared discrepancy $a$), so $\beta$ is identifiable.

**Noise inflation** and **additive $a\cdot g$ GP** share
$$
a(t,L)=\Bigl(\frac{|t|}{t_0}\Bigr)^p+\Bigl(\frac{L_0}{L}\Bigr)^q.
$$

**Noise inflation** (legacy):
$$\sigma_{\mathrm{eff}}=\sqrt{\sigma_{\mathrm{MC}}^2+(\sigma_{\mathrm{model}}\,K\,a)^2}.$$

**Additive GP** (slide form; $\Phi_1$ dropped, $r=g$):
$$m=L^{-\beta/\nu}f(z)+a\,g(z)+\varepsilon_m
\quad\Rightarrow\quad
\Phi_m=f+(a\,L^{\beta/\nu})g+\varepsilon_\Phi,$$
and $\Phi_{m^2}=f+(a\,L^{2\beta/\nu})g+\varepsilon$ (shared raw $a$).  Independent GPs
$f\sim\mathrm{GP}(0,\eta^2 k)$, $g\sim\mathrm{GP}(0,\sigma_g^2 k)$.
Slider **σ_g** (same control as σ_model) sets the discrepancy GP amplitude; try large values to absorb gated points.

**Z-threshold additive GP**: collapsed amplitude
$a=1_{|z|>10}$ (else $0$), no $L^{\beta/\nu}$ factor.  Large $\sigma_g$ should nearly remove those points' pull on $f$ and $\nu$.

**K** scales discrepancy ($K=0$ off; $K=1$ default).

- **Top-left:** $\Phi_m$ vs $z$ at exact exponents (bars = $\sigma_{\mathrm{eff}}$ or $\sigma_{\mathrm{MC}}$)
- **Heatmaps:** $(T_c,\nu)$ at fixed $\beta$; $(\nu,\beta)$ at fixed $T_c$; $(T_c,\beta)$ at fixed $\nu$

**Selection (collapse plot):** legend click includes/excludes all points of that `L` and recomputes the heatmap. Click a marker to toggle one point (grey = excluded). Box/lasso-drag to toggle a group. Use **Include all points** to reset.

Use the **dataset** and **model** dropdowns. Changing either recompiles the JAX likelihood.


In [1]:
from __future__ import annotations

import sys
import time
from pathlib import Path

import numpy as np
import ipywidgets as widgets
import plotly.graph_objects as go
from IPython.display import display

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from ising.constants import (
    BETA_EXACT as ISING_BETA_EXACT,
    NU_EXACT as ISING_NU_EXACT,
    TC_EXACT as ISING_TC_EXACT,
)
from ising.datasets import (
    get_dataset as get_ising_dataset,
    list_datasets as list_ising_datasets,
    observables_path as ising_observables_path,
)
from phi4 import fss_io as phi4_fss
from phi4.datasets import (
    get_dataset as get_phi4_dataset,
    list_datasets as list_phi4_datasets,
    observables_path as phi4_observables_path,
)
from ising.discrepancy import (
    DISCREPANCY_L0_INIT,
    DISCREPANCY_L0_PRIOR_LOWER,
    DISCREPANCY_L0_PRIOR_UPPER,
    DISCREPANCY_P_INIT,
    DISCREPANCY_P_PRIOR_LOWER,
    DISCREPANCY_P_PRIOR_UPPER,
    DISCREPANCY_Q_INIT,
    DISCREPANCY_Q_PRIOR_LOWER,
    DISCREPANCY_Q_PRIOR_UPPER,
    DISCREPANCY_SIGMA_MODEL_INIT,
    DISCREPANCY_T0_INIT,
    DISCREPANCY_T0_PRIOR_LOWER,
    DISCREPANCY_T0_PRIOR_UPPER,
    discrepancy_amplitude,
    effective_obs_sigma,
)
from ising.fss_likelihood import (
    compile_fss_log_marginal_likelihood_jax,
    profile_joint_tc_nu_with_disc,
)
from ising.gp_jax import (
    DEFAULT_GP_KERNEL,
    discrepancy_f_posterior_mean,
    gp_posterior_predictive,
)
from ising.jax_config import configure_jax, jax_device_summary
from ising.model_fss import NU_PRIOR_LOWER, NU_PRIOR_UPPER
from ising.model_scaling_nu import (
    TC_PRIOR_LOWER as ISING_TC_PRIOR_LOWER,
    TC_PRIOR_UPPER as ISING_TC_PRIOR_UPPER,
)
from ising.observables import read_observables_for_fss
from ising.profile_likelihood import (
    FssProfileConfig,
    Z_DISC_THRESHOLD,
)
from ising.traditional_scaling import magnetization_collapse_arrays

# Active truth / prior window (switched when loading Ising vs φ⁴ datasets).
TC_EXACT = ISING_TC_EXACT
NU_EXACT = ISING_NU_EXACT
BETA_EXACT = ISING_BETA_EXACT
TC_PRIOR_LOWER = ISING_TC_PRIOR_LOWER
TC_PRIOR_UPPER = ISING_TC_PRIOR_UPPER

configure_jax()
print(jax_device_summary())


backend=cpu, devices=[cpu:0]


In [2]:
DEFAULT_FAMILY = "ising"
DEFAULT_DATASET = "harada_square_large_t"
DEFAULT_DISC_FORM = "noise"  # or "additive_gp" / "additive_gp_z_threshold"
HEATMAP_N = 35
HEATMAP_Z_FLOOR = -80.0
HEATMAP_ZOOM_LEVELS = (-2.0, -6.0, -20.0, -80.0)
HEATMAP_MIN_ZOOM_CELLS = 4
COLLAPSE_WIDTH = 520
COLLAPSE_HEIGHT = 420
HEATMAP_WIDTH = 480
HEATMAP_HEIGHT = 420
MIN_INCLUDED_POINTS = 4
SIGMA_SLIDER_MAX = 20.0

PLOTLY_COLORS = (
    "#1f77b4",
    "#ff7f0e",
    "#2ca02c",
    "#d62728",
    "#9467bd",
    "#8c564b",
    "#e377c2",
    "#7f7f7f",
)
EXCLUDED_COLOR = "#b0b0b0"

DISC_FORM_OPTIONS = (
    ("noise inflation (σ_eff)", "noise"),
    ("additive a·g GP", "additive_gp"),
    ("a=1 if |z|>10 else 0", "additive_gp_z_threshold"),
)

DISC_FORM_LABELS = {
    "noise": "noise inflation",
    "additive_gp": "additive a·g GP",
    "additive_gp_z_threshold": "a=1_{|z|>10} GP",
}


def _is_additive_form(form: str | None = None) -> bool:
    f = state["disc_form"] if form is None else form
    return f in ("additive_gp", "additive_gp_z_threshold")


def _form_tag(form: str | None = None) -> str:
    f = state["disc_form"] if form is None else form
    if f == "additive_gp":
        return "a·g"
    if f == "additive_gp_z_threshold":
        return "|z|>10"
    return "σ_eff"


def make_config(discrepancy_form: str) -> FssProfileConfig:
    return FssProfileConfig(
        use_m=True,
        use_m2=True,
        use_m4=False,
        use_binder=False,
        use_chi=False,
        correction_m=False,
        correction_m2=False,
        correction_m4=False,
        correction_binder=False,
        correction_chi=False,
        discrepancy_m=True,
        discrepancy_m2=True,
        discrepancy_form=discrepancy_form,  # type: ignore[arg-type]
        z_disc_threshold=float(Z_DISC_THRESHOLD),
        use_log_m=False,
        gp_kernel="gaussian",
        # Absolute length scale in z (not factor × data z_span).
        gp_ell=0.15,
        correction_gp_ell=0.15,
        gp_eta=1.0,
    )


def _profile_grid(lo: float, hi: float, exact: float, n: int) -> np.ndarray:
    grid = np.linspace(lo, hi, n)
    if lo <= exact <= hi and not np.any(np.isclose(grid, exact, rtol=0.0, atol=1e-10)):
        grid = np.sort(np.append(grid, exact))
    return grid


tc_grid = _profile_grid(TC_PRIOR_LOWER, TC_PRIOR_UPPER, TC_EXACT, HEATMAP_N)
nu_grid = _profile_grid(NU_PRIOR_LOWER, NU_PRIOR_UPPER, NU_EXACT, HEATMAP_N)


def _observables_path(family: str, name: str) -> Path:
    if family == "ising":
        return ising_observables_path(name)
    if family == "phi4":
        return phi4_observables_path(name)
    raise ValueError(f"Unknown dataset family {family!r}")


def _get_dataset(family: str, name: str):
    if family == "ising":
        return get_ising_dataset(name)
    if family == "phi4":
        return get_phi4_dataset(name)
    raise ValueError(f"Unknown dataset family {family!r}")


def _dataset_has_m2(family: str, name: str) -> bool:
    path = _observables_path(family, name)
    if not path.is_file():
        return False
    header = path.read_text().splitlines()[0]
    cols = {c.strip() for c in header.split(",")}
    return {"m2", "m2_std", "n_eff_m2"}.issubset(cols) or (
        "magnetization" in cols and "m2" in cols
    )


def _set_truth_for_family(family: str) -> None:
    """Switch active exact exponents / Tc prior window (mutates module globals)."""
    global TC_EXACT, NU_EXACT, BETA_EXACT, TC_PRIOR_LOWER, TC_PRIOR_UPPER, tc_grid, nu_grid
    if family == "ising":
        TC_EXACT = ISING_TC_EXACT
        NU_EXACT = ISING_NU_EXACT
        BETA_EXACT = ISING_BETA_EXACT
        TC_PRIOR_LOWER = ISING_TC_PRIOR_LOWER
        TC_PRIOR_UPPER = ISING_TC_PRIOR_UPPER
    elif family == "phi4":
        TC_EXACT, NU_EXACT, BETA_EXACT = phi4_fss.truth_exponents()
        TC_PRIOR_LOWER = phi4_fss.TC_PRIOR_LOWER
        TC_PRIOR_UPPER = phi4_fss.TC_PRIOR_UPPER
    else:
        raise ValueError(f"Unknown dataset family {family!r}")
    tc_grid = _profile_grid(TC_PRIOR_LOWER, TC_PRIOR_UPPER, TC_EXACT, HEATMAP_N)
    nu_grid = _profile_grid(NU_PRIOR_LOWER, NU_PRIOR_UPPER, NU_EXACT, HEATMAP_N)


AVAILABLE_DATASETS: list[tuple[str, tuple[str, str]]] = []
for _name in list_ising_datasets():
    if _dataset_has_m2("ising", _name):
        AVAILABLE_DATASETS.append((f"ising/{_name}", ("ising", _name)))
for _name in list_phi4_datasets():
    if _dataset_has_m2("phi4", _name):
        AVAILABLE_DATASETS.append((f"phi4/{_name}", ("phi4", _name)))

_default_key = (DEFAULT_FAMILY, DEFAULT_DATASET)
if not any(val == _default_key for _, val in AVAILABLE_DATASETS):
    raise FileNotFoundError(
        f"Default dataset {DEFAULT_FAMILY}/{DEFAULT_DATASET!r} CSV missing; "
        f"available: {[lab for lab, _ in AVAILABLE_DATASETS]}"
    )

state: dict = {
    "family": None,
    "dataset": None,
    "disc_form": None,
    "config": None,
    "df_full": None,
    "df": None,
    "included": None,
    "compiled": None,
    "z0": None,
    "phi0": None,
    "sigma_phi_mc": None,
    "L_arr": None,
    "t0_arr": None,
    "T_arr": None,
    "L_values": (),
    "Y_LO": 0.0,
    "Y_HI": 1.0,
}


def _active_df():
    mask = np.asarray(state["included"], dtype=bool)
    return state["df_full"].iloc[np.flatnonzero(mask)].reset_index(drop=True)


def _recompile_active(*, quiet: bool = True) -> bool:
    """Recompile JAX likelihood on currently included points. Returns False if too few."""
    mask = np.asarray(state["included"], dtype=bool)
    n_in = int(mask.sum())
    if n_in < MIN_INCLUDED_POINTS:
        return False
    df_active = state["df_full"].iloc[np.flatnonzero(mask)].reset_index(drop=True)
    compiled = compile_fss_log_marginal_likelihood_jax(df_active, state["config"])
    state["df"] = df_active
    state["compiled"] = compiled
    if not quiet:
        print(f"Recompiled on {n_in}/{len(mask)} included points")
    return True


def load_dataset(
    name: str,
    *,
    family: str | None = None,
    discrepancy_form: str | None = None,
    quiet: bool = False,
) -> None:
    """Load observables, compile likelihood, and refresh collapse arrays."""
    fam = DEFAULT_FAMILY if family is None else str(family)
    form = DEFAULT_DISC_FORM if discrepancy_form is None else str(discrepancy_form)
    _set_truth_for_family(fam)
    config = make_config(form)
    data_path = _observables_path(fam, name)
    if fam == "phi4":
        df = phi4_fss.load_observables(name, path=data_path)
    else:
        df = read_observables_for_fss(data_path, use_m=True, use_m2=True, use_binder=False)
    included = np.ones(len(df), dtype=bool)
    z0, phi0, sigma_phi_mc, L_arr = magnetization_collapse_arrays(
        df, TC_EXACT, NU_EXACT, BETA_EXACT
    )
    T_arr = df["T"].to_numpy(dtype=np.float64)
    t0_arr = (T_arr - TC_EXACT) / TC_EXACT
    L_values = tuple(sorted(int(v) for v in np.unique(L_arr)))
    y_pad = 0.08 * float(np.ptp(phi0) + 1e-12)
    state.update(
        family=fam,
        dataset=name,
        disc_form=form,
        config=config,
        df_full=df,
        included=included,
        z0=z0,
        phi0=phi0,
        sigma_phi_mc=sigma_phi_mc,
        L_arr=L_arr,
        t0_arr=t0_arr,
        T_arr=T_arr,
        L_values=L_values,
        Y_LO=float(np.min(phi0 - sigma_phi_mc)) - y_pad,
        Y_HI=float(np.max(phi0 + sigma_phi_mc)) + y_pad,
    )
    if not _recompile_active(quiet=True):
        raise RuntimeError(f"Dataset {fam}/{name!r} has fewer than {MIN_INCLUDED_POINTS} points")
    if not quiet:
        spec = _get_dataset(fam, name)
        form_label = DISC_FORM_LABELS.get(form, form)
        print(
            f"Loaded {fam}/{name}: {len(df)} points from {data_path}\n"
            f"  {spec.description}\n"
            f"Model: {form_label}; channels=m+m²; "
            f"gp_ell={state['compiled'].gp_scales.base_gp_ell:.4g}, "
            f"gp_eta={state['compiled'].gp_scales.base_gp_eta:.4g}; "
            f"L={L_values}; heatmap {tc_grid.size}×{nu_grid.size}; "
            f"Tc_exact={TC_EXACT:.5g}"
        )


load_dataset(DEFAULT_DATASET, family=DEFAULT_FAMILY, discrepancy_form=DEFAULT_DISC_FORM)


Loaded ising/harada_square_large_t: 151 points from /home/reuben/HMCLib/ising/data/observables_harada_square_large_t.csv
  Harada square Binder grid (L=64/128/256) plus small-L×large-|t| points (L=8–32, T∈[1.70,8]) to expose FSS collapse failure away from the joint t→0, L→∞ limit.
Model: noise inflation; channels=m+m²; gp_ell=13.33, gp_eta=1; L=(8, 12, 16, 24, 32, 64, 128, 256); heatmap 36×35; Tc_exact=2.2692


In [3]:
def _disc_params() -> dict[str, float]:
    return {
        "t0": float(slider_t0.value),
        "L0": float(slider_L0.value),
        "p": float(slider_p.value),
        "q": float(slider_q.value),
        "sigma_model": float(slider_sigma.value),
        "K": float(slider_K.value),
    }


def _n_included() -> int:
    return int(np.asarray(state["included"], dtype=bool).sum())


def _inclusion_status_text() -> str:
    n_tot = len(state["included"])
    n_in = _n_included()
    return f"included {n_in}/{n_tot}"


def _inclusion_status_html() -> str:
    n_tot = len(state["included"])
    n_in = _n_included()
    return f"included <b>{n_in}</b>/{n_tot}"


def _sigma_eff_phi(params: dict[str, float]) -> tuple[np.ndarray, np.ndarray]:
    """Return (K·a_vis, sigma_plot). Additive forms: bars are MC-only.

    K scales the discrepancy. For z-threshold, a_vis = 1_{|z|>thr} at exact
    collapse z (display only; heatmap recomputes a from z(Tc,ν)).
    """
    if state["disc_form"] == "additive_gp_z_threshold":
        thr = float(getattr(state["config"], "z_disc_threshold", Z_DISC_THRESHOLD))
        a = (np.abs(np.asarray(state["z0"], dtype=np.float64)) > thr).astype(
            np.float64
        )
        Ka = float(params["K"]) * a
        return Ka, np.asarray(state["sigma_phi_mc"], dtype=np.float64)

    a = discrepancy_amplitude(
        state["t0_arr"],
        state["L_arr"],
        t0=params["t0"],
        L0=params["L0"],
        p=params["p"],
        q=params["q"],
    )
    Ka = float(params["K"]) * a
    if state["disc_form"] == "additive_gp":
        return Ka, np.asarray(state["sigma_phi_mc"], dtype=np.float64)
    sigma_eff = effective_obs_sigma(
        state["sigma_phi_mc"],
        state["t0_arr"],
        state["L_arr"],
        t0=params["t0"],
        L0=params["L0"],
        p=params["p"],
        q=params["q"],
        sigma_model=params["sigma_model"] * float(params["K"]),
    )
    return Ka, sigma_eff



def _f_hat_phi_m(params: dict[str, float]) -> np.ndarray:
    """GP posterior mean of universal f at exact-(Tc,ν,β) collapse points.

    Trained on currently included points only (same mask as the heatmap LML).
    Noise model: latent GP with σ_eff. Additive GP: E[f|y] from
    Φ = f + (a L^{β/ν}) g + ε (K absorbed into σ_g, matching the heatmap).
    """
    included = np.asarray(state["included"], dtype=bool)
    z = np.asarray(state["z0"], dtype=np.float64)
    phi = np.asarray(state["phi0"], dtype=np.float64)
    n = z.size
    out = np.full(n, np.nan, dtype=np.float64)
    if int(included.sum()) < 2:
        return out

    a_full = discrepancy_amplitude(
        state["t0_arr"],
        state["L_arr"],
        t0=params["t0"],
        L0=params["L0"],
        p=params["p"],
        q=params["q"],
    )
    sigma_mc = np.asarray(state["sigma_phi_mc"], dtype=np.float64)
    ell = float(state["compiled"].gp_scales.base_gp_ell)
    eta = float(state["compiled"].gp_scales.base_gp_eta)
    kernel = getattr(state["config"], "gp_kernel", None) or DEFAULT_GP_KERNEL

    z_tr = z[included]
    y_tr = phi[included]
    if state["disc_form"] == "additive_gp":
        # Slide: m = L^{-β/ν} f + a g  ⇒  Φ = f + (a L^{β/ν}) g.
        # K·a·g ≡ a·g' with g' ~ GP(0, (K σ_g)² k), same as heatmap LML.
        L_arr = np.asarray(state["L_arr"], dtype=np.float64)
        a_collapsed = a_full * (L_arr ** (BETA_EXACT / NU_EXACT))
        sigma_tr = sigma_mc[included]
        a_tr = a_collapsed[included]
        mean = discrepancy_f_posterior_mean(
            z_tr,
            y_tr,
            sigma_tr,
            a_tr,
            z,
            gp_ell=ell,
            gp_eta=eta,
            disc_gp_ell=ell,
            disc_gp_eta=float(params["sigma_model"]) * float(params["K"]),
            kernel=kernel,
        )
    elif state["disc_form"] == "additive_gp_z_threshold":
        thr = float(getattr(state["config"], "z_disc_threshold", Z_DISC_THRESHOLD))
        a_collapsed = (np.abs(z) > thr).astype(np.float64)
        mean = discrepancy_f_posterior_mean(
            z_tr,
            y_tr,
            sigma_mc[included],
            a_collapsed[included],
            z,
            gp_ell=ell,
            gp_eta=eta,
            disc_gp_ell=ell,
            disc_gp_eta=float(params["sigma_model"]) * float(params["K"]),
            kernel=kernel,
        )
    else:
        sigma_eff = effective_obs_sigma(
            sigma_mc,
            state["t0_arr"],
            state["L_arr"],
            t0=params["t0"],
            L0=params["L0"],
            p=params["p"],
            q=params["q"],
            sigma_model=params["sigma_model"] * float(params["K"]),
        )
        mean, _ = gp_posterior_predictive(
            z_tr,
            y_tr,
            sigma_eff[included],
            z,
            length_scale=ell,
            amplitude=eta,
            kernel=kernel,
        )
    return np.asarray(mean, dtype=np.float64)


def _heatmap_axis_ranges(
    z: np.ndarray, tc_mle: float, nu_mle: float, dlog_exact: float
) -> tuple[list[float], list[float], float, int, bool]:
    tc_span = float(tc_grid[-1] - tc_grid[0])
    nu_span = float(nu_grid[-1] - nu_grid[0])
    d_tc = float(np.min(np.diff(tc_grid)))
    d_nu = float(np.min(np.diff(nu_grid)))

    level_used = HEATMAP_Z_FLOOR
    n_cells = 0
    mask = None
    for level in HEATMAP_ZOOM_LEVELS:
        cand = z > level
        n = int(np.count_nonzero(cand))
        if n >= HEATMAP_MIN_ZOOM_CELLS:
            mask = cand
            level_used = float(level)
            n_cells = n
            break
        if n > n_cells:
            mask = cand
            level_used = float(level)
            n_cells = n

    if mask is not None and n_cells >= 1:
        jj, ii = np.where(mask)
        tc_lo = float(tc_grid[ii.min()])
        tc_hi = float(tc_grid[ii.max()])
        nu_lo = float(nu_grid[jj.min()])
        nu_hi = float(nu_grid[jj.max()])
    else:
        tc_lo = tc_hi = tc_mle
        nu_lo = nu_hi = nu_mle

    tc_lo = min(tc_lo, tc_mle)
    tc_hi = max(tc_hi, tc_mle)
    nu_lo = min(nu_lo, nu_mle)
    nu_hi = max(nu_hi, nu_mle)
    include_exact = bool(np.isfinite(dlog_exact) and dlog_exact > HEATMAP_Z_FLOOR)
    if include_exact:
        tc_lo = min(tc_lo, TC_EXACT)
        tc_hi = max(tc_hi, TC_EXACT)
        nu_lo = min(nu_lo, NU_EXACT)
        nu_hi = max(nu_hi, NU_EXACT)

    pad_tc = max(2.5 * d_tc, 0.04 * tc_span)
    pad_nu = max(2.5 * d_nu, 0.04 * nu_span)
    min_half_tc = 0.06 * tc_span
    min_half_nu = 0.06 * nu_span
    tc_c = 0.5 * (tc_lo + tc_hi)
    nu_c = 0.5 * (nu_lo + nu_hi)
    half_tc = max(0.5 * (tc_hi - tc_lo) + pad_tc, min_half_tc)
    half_nu = max(0.5 * (nu_hi - nu_lo) + pad_nu, min_half_nu)

    x_range = [
        max(float(tc_grid[0]), tc_c - half_tc),
        min(float(tc_grid[-1]), tc_c + half_tc),
    ]
    y_range = [
        max(float(nu_grid[0]), nu_c - half_nu),
        min(float(nu_grid[-1]), nu_c + half_nu),
    ]
    exact_in_view = include_exact and (
        x_range[0] <= TC_EXACT <= x_range[1] and y_range[0] <= NU_EXACT <= y_range[1]
    )
    return x_range, y_range, level_used, n_cells, exact_in_view


def _marker_style_for_L(
    Lval: int,
    order: np.ndarray,
    a: np.ndarray,
    *,
    color: str,
) -> tuple[list[float], list[float], list[str]]:
    """Return (sizes, opacities, colors) for one L-trace in display order."""
    included = np.asarray(state["included"], dtype=bool)
    L_arr = state["L_arr"]
    mask = L_arr == Lval
    idxs = np.flatnonzero(mask)[order]
    a_L = a[mask][order]
    size = (7.0 + 10.0 * np.clip(a_L / (1.0 + a_L), 0.0, 1.0)).tolist()
    opacity = []
    colors = []
    for j, a_j in zip(idxs, a_L):
        if included[j]:
            colors.append(color)
            opacity.append(float(np.clip(1.0 / (1.0 + 0.5 * a_j), 0.35, 0.95)))
        else:
            colors.append(EXCLUDED_COLOR)
            opacity.append(0.25)
    return size, opacity, colors


def _build_collapse_figure() -> go.FigureWidget:
    params = {
        "t0": DISCREPANCY_T0_INIT,
        "L0": DISCREPANCY_L0_INIT,
        "p": DISCREPANCY_P_INIT,
        "q": DISCREPANCY_Q_INIT,
        "sigma_model": DISCREPANCY_SIGMA_MODEL_INIT,
        "K": 1.0,
    }
    a, sigma_plot = _sigma_eff_phi(params)
    f_hat = _f_hat_phi_m(params)
    z0 = state["z0"]
    phi0 = state["phi0"]
    L_arr = state["L_arr"]
    T_arr = state["T_arr"]
    L_values = state["L_values"]
    Y_LO = state["Y_LO"]
    Y_HI = state["Y_HI"]
    included = np.asarray(state["included"], dtype=bool)
    traces = []
    for i, Lval in enumerate(L_values):
        mask = L_arr == Lval
        order = np.argsort(z0[mask])
        idxs = np.flatnonzero(mask)[order]
        color = PLOTLY_COLORS[i % len(PLOTLY_COLORS)]
        size, opacity, colors = _marker_style_for_L(Lval, order, a, color=color)
        n_in_L = int(included[mask].sum())
        visible = True if n_in_L > 0 else "legendonly"
        traces.append(
            go.Scatter(
                x=z0[mask][order].tolist(),
                y=phi0[mask][order].tolist(),
                customdata=np.column_stack(
                    [
                        idxs,
                        T_arr[mask][order],
                        a[mask][order],
                        f_hat[mask][order],
                        (phi0[mask][order] - f_hat[mask][order]),
                    ]
                ).tolist(),
                error_y=dict(
                    type="data",
                    array=sigma_plot[mask][order].tolist(),
                    visible=True,
                    thickness=1.0,
                    width=2,
                ),
                mode="markers",
                marker=dict(size=size, color=colors, opacity=opacity),
                name=f"L={Lval}",
                visible=visible,
                hovertemplate=(
                    f"idx=%{{customdata[0]:.0f}}<br>L={Lval}"
                    "<br>T=%{customdata[1]:.4f}<br>z=%{x:.3g}<br>Φ=%{y:.4f}"
                    "<br>f=%{customdata[3]:.4f}<br>Φ−f=%{customdata[4]:.4f}"
                    "<br>a=%{customdata[2]:.3g}<extra></extra>"
                ),
            )
        )
    traces.append(
        go.Scatter(
            x=[0.0, 0.0],
            y=[Y_LO, Y_HI],
            mode="lines",
            line=dict(color="gray", width=1, dash="dot"),
            showlegend=False,
            hoverinfo="skip",
        )
    )
    bar_label = "σ_MC" if _is_additive_form() else "σ_eff"
    fig = go.FigureWidget(
        data=traces,
        layout=go.Layout(
            title=dict(
                text=(
                    f"m collapse (LML: m+m²) [{state['dataset']}] ({bar_label} bars) · "
                    f"{_inclusion_status_text()}"
                ),
                font=dict(size=13),
            ),
            xaxis_title="z = t L^(1/ν)",
            yaxis_title="Φ_m = |m| L^(β/ν)",
            yaxis=dict(range=[Y_LO, Y_HI]),
            width=COLLAPSE_WIDTH,
            height=COLLAPSE_HEIGHT,
            template="plotly_white",
            margin=dict(l=56, r=16, t=48, b=48),
            legend=dict(
                font=dict(size=9),
                yanchor="top",
                y=0.99,
                xanchor="right",
                x=0.99,
                itemclick="toggle",
                itemdoubleclick="toggleothers",
            ),
            # Box/lasso select toggles inclusion of selected points.
            dragmode="select",
            clickmode="event+select",
            selectdirection="any",
            hovermode="closest",
        ),
    )
    _bind_collapse_interactions(fig)
    return fig


def _build_heatmap_figure() -> go.FigureWidget:
    z0 = np.full((nu_grid.size, tc_grid.size), HEATMAP_Z_FLOOR, dtype=np.float64).tolist()
    return go.FigureWidget(
        data=[
            go.Heatmap(
                x=np.asarray(tc_grid, dtype=np.float64).tolist(),
                y=np.asarray(nu_grid, dtype=np.float64).tolist(),
                z=z0,
                zmin=HEATMAP_Z_FLOOR,
                zmax=0.0,
                colorscale="Viridis",
                colorbar=dict(title="ΔlogL", thickness=14, len=0.8),
                hovertemplate="Tc=%{x:.5f}<br>ν=%{y:.4f}<br>ΔlogL=%{z:.2f}<extra></extra>",
            ),
            go.Contour(
                x=np.asarray(tc_grid, dtype=np.float64).tolist(),
                y=np.asarray(nu_grid, dtype=np.float64).tolist(),
                z=z0,
                showscale=False,
                contours=dict(start=-20, end=0, size=2, coloring="lines"),
                line=dict(width=0.6, color="rgba(255,255,255,0.35)"),
                hoverinfo="skip",
            ),
            go.Scatter(
                x=[TC_EXACT],
                y=[NU_EXACT],
                mode="markers",
                marker=dict(
                    symbol="circle",
                    size=11,
                    color="white",
                    line=dict(width=1.2, color="black"),
                ),
                name="exact",
                hovertemplate=f"exact Tc={TC_EXACT:.5f}, ν={NU_EXACT:g}<extra></extra>",
            ),
            go.Scatter(
                x=[TC_EXACT],
                y=[NU_EXACT],
                mode="markers",
                marker=dict(
                    symbol="x",
                    size=12,
                    color="#c44e52",
                    line=dict(width=1.5, color="white"),
                ),
                name="grid MLE",
                hovertemplate="MLE Tc=%{x:.5f}, ν=%{y:.4f}<extra></extra>",
            ),
        ],
        layout=go.Layout(
            title=dict(text="(Tc, ν) log-marginal (β fixed)", font=dict(size=13)),
            xaxis_title="Tc",
            yaxis_title="ν",
            xaxis=dict(range=[float(tc_grid[0]), float(tc_grid[-1])]),
            yaxis=dict(range=[float(nu_grid[0]), float(nu_grid[-1])]),
            width=HEATMAP_WIDTH,
            height=HEATMAP_HEIGHT,
            template="plotly_white",
            margin=dict(l=48, r=72, t=48, b=48),
            legend=dict(font=dict(size=9), yanchor="top", y=0.99, xanchor="left", x=0.01),
        ),
    )


def _update_collapse(params: dict[str, float]) -> None:
    a, sigma_plot = _sigma_eff_phi(params)
    f_hat = _f_hat_phi_m(params)
    z0 = state["z0"]
    phi0 = state["phi0"]
    L_arr = state["L_arr"]
    T_arr = state["T_arr"]
    L_values = state["L_values"]
    Y_LO = state["Y_LO"]
    Y_HI = state["Y_HI"]
    included = np.asarray(state["included"], dtype=bool)
    global _updating_from_code
    n_clip = int(np.sum((phi0 - sigma_plot < Y_LO) | (phi0 + sigma_plot > Y_HI)))
    bar_label = "σ_MC" if _is_additive_form() else "σ_eff"
    _updating_from_code = True
    try:
        with collapse_fig.batch_update():
            for i, Lval in enumerate(L_values):
                mask = L_arr == Lval
                order = np.argsort(z0[mask])
                idxs = np.flatnonzero(mask)[order]
                color = PLOTLY_COLORS[i % len(PLOTLY_COLORS)]
                size, opacity, colors = _marker_style_for_L(Lval, order, a, color=color)
                tr = collapse_fig.data[i]
                # FigureWidget requires plain Python lists (numpy arrays break
                # plotly.basewidget._remove_overlapping_props via `if not arr`).
                tr.x = z0[mask][order].tolist()
                tr.y = phi0[mask][order].tolist()
                tr.customdata = np.column_stack(
                    [
                        idxs,
                        T_arr[mask][order],
                        a[mask][order],
                        f_hat[mask][order],
                        (phi0[mask][order] - f_hat[mask][order]),
                    ]
                ).tolist()
                tr.error_y.array = sigma_plot[mask][order].tolist()
                tr.marker.size = list(size)
                tr.marker.opacity = list(opacity)
                tr.marker.color = list(colors)
                # Keep legend visibility in sync with whether any L points are included.
                n_in_L = int(included[mask].sum())
                want_visible: bool | str = True if n_in_L > 0 else "legendonly"
                if tr.visible != want_visible:
                    tr.visible = want_visible
            guide = collapse_fig.data[len(L_values)]
            guide.y = [Y_LO, Y_HI]
            collapse_fig.layout.yaxis.range = [Y_LO, Y_HI]
            collapse_fig.layout.title.text = (
                f"m collapse (LML: m+m²) [{state['dataset']}] + {bar_label} "
                f"(K={params['K']:.3g}, t0={params['t0']:.3g}, L0={params['L0']:.3g}, "
                f"p={params['p']:.3g}, q={params['q']:.3g}, "
                f"σ={params['sigma_model']:.3g}; "
                f"{n_clip}/{len(phi0)} bars clipped; {_inclusion_status_text()})"
            )
    finally:
        _updating_from_code = False


def _update_heatmap(params: dict[str, float]) -> None:
    if state["compiled"] is None or _n_included() < MIN_INCLUDED_POINTS:
        status.value = (
            f"<span style='color:#c44e52'>Need ≥{MIN_INCLUDED_POINTS} included points "
            f"({_inclusion_status_html()}). Click legend / points to include more.</span>"
        )
        return
    t_start = time.perf_counter()
    # Effective disc amplitude: Φ = f + (K a) g with g~GP(0, σ_g² k)
    # is implemented as disc_gp_eta = K·σ_g (same for noise σ_model).
    k_scale = float(params["K"])
    sigma_slider = float(params["sigma_model"])
    sigma_eff = sigma_slider * k_scale
    log_ml = profile_joint_tc_nu_with_disc(
        state["compiled"],
        tc_grid,
        nu_grid,
        beta=BETA_EXACT,
        t0=params["t0"],
        L0=params["L0"],
        p=params["p"],
        q=params["q"],
        sigma_model=sigma_eff,
    )
    z_raw = np.asarray(log_ml, dtype=np.float64)
    z_max = float(np.nanmax(z_raw))
    z = z_raw - z_max
    z_plot = np.maximum(z, HEATMAP_Z_FLOOR)

    j_max, i_max = np.unravel_index(int(np.nanargmax(z)), z.shape)
    tc_mle = float(tc_grid[i_max])
    nu_mle = float(nu_grid[j_max])

    i_exact = int(np.argmin(np.abs(tc_grid - TC_EXACT)))
    j_exact = int(np.argmin(np.abs(nu_grid - NU_EXACT)))
    dlog_exact = float(z[j_exact, i_exact])

    x_range, y_range, zoom_level, n_zoom, exact_in_view = _heatmap_axis_ranges(
        z_plot, tc_mle, nu_mle, dlog_exact
    )

    in_x = (tc_grid >= x_range[0]) & (tc_grid <= x_range[1])
    in_y = (nu_grid >= y_range[0]) & (nu_grid <= y_range[1])
    z_view = z_plot[np.ix_(in_y, in_x)]
    above_floor = z_view[z_view > HEATMAP_Z_FLOOR + 1e-9]
    if above_floor.size >= 2:
        zmin_view = float(max(HEATMAP_Z_FLOOR, np.nanmin(above_floor)))
    elif above_floor.size == 1:
        zmin_view = float(max(HEATMAP_Z_FLOOR, above_floor[0] - 5.0))
    else:
        zmin_view = HEATMAP_Z_FLOOR

    elapsed = time.perf_counter() - t_start
    form_tag = _form_tag()
    sigma_name = "σ_g" if _is_additive_form() else "σ_model"
    k_warn = (
        " <span style='color:#c44e52'><b>K=0 ⇒ discrepancy off</b> (σ slider ignored)</span>"
        if k_scale <= 0.0
        else ""
    )
    with heatmap_fig.batch_update():
        heatmap_fig.data[0].z = z_plot.tolist()
        heatmap_fig.data[0].zmin = zmin_view
        heatmap_fig.data[0].zmax = 0.0
        heatmap_fig.data[1].z = z_plot.tolist()
        heatmap_fig.data[2].x = [TC_EXACT]
        heatmap_fig.data[2].y = [NU_EXACT]
        heatmap_fig.data[3].x = [tc_mle]
        heatmap_fig.data[3].y = [nu_mle]
        heatmap_fig.layout.xaxis.range = x_range
        heatmap_fig.layout.yaxis.range = y_range
        heatmap_fig.layout.title.text = (
            f"[{state['dataset']}|{form_tag}|n={_n_included()}] "
            f"{sigma_name}={sigma_slider:.3g} K={k_scale:.3g} → η_disc={sigma_eff:.3g} · "
            f"MLE Tc={tc_mle:.5f}, ν={nu_mle:.4g} · "
            f"exact Δ={dlog_exact:.1f} maxL={z_max:.1f} ({elapsed:.2f}s)"
        )
    sharp = (
        "sharp peak — zoomed"
        if n_zoom < HEATMAP_MIN_ZOOM_CELLS
        else f"zoom ΔlogL>{zoom_level:g}"
    )
    exact_note = "exact in view" if exact_in_view else "exact off-zoom (far below peak)"
    status.value = (
        f"<b>{state['dataset']}</b> · <b>{form_tag}</b> · form=<code>{state['disc_form']}</code> · "
        f"{_inclusion_status_html()} · "
        f"{sigma_name}={sigma_slider:.3g} · K={k_scale:.3g} · η_disc={sigma_eff:.3g} · "
        f"heatmap {elapsed:.2f}s · "
        f"MLE Tc={tc_mle:.6f}, ν={nu_mle:.5f} · "
        f"ΔlogL(exact)={dlog_exact:.2f} · maxL={z_max:.1f} ({exact_note}) · "
        f"{sharp} ({n_zoom} cells){k_warn}"
    )


def _apply_inclusion_change(*, refresh_collapse: bool = True) -> None:
    """Recompile + refresh heatmap (and optionally collapse styling) after mask edits."""
    params = _disc_params()
    if not _recompile_active(quiet=True):
        status.value = (
            f"<span style='color:#c44e52'>Need ≥{MIN_INCLUDED_POINTS} included points "
            f"({_inclusion_status_html()}).</span>"
        )
        if refresh_collapse:
            _update_collapse(params)
        return
    status.value = f"updating heatmap ({_inclusion_status_html()})…"
    if refresh_collapse:
        _update_collapse(params)
    _update_heatmap(params)


# --- interactive selection on collapse plot ---
_updating_from_code = False


def _trace_is_visible(visible) -> bool:
    return visible is True or visible is None


def _on_legend_visibility_change(trace, points_unused=None) -> None:
    """Legend click hides/shows an L-trace → include/exclude all points of that L."""
    global _updating_from_code
    if _updating_from_code:
        return
    # Map trace → L via name "L=..."
    name = str(getattr(trace, "name", "") or "")
    if not name.startswith("L="):
        return
    try:
        Lval = int(name.split("=", 1)[1])
    except ValueError:
        return
    include = _trace_is_visible(trace.visible)
    L_arr = state["L_arr"]
    mask = L_arr == Lval
    included = np.asarray(state["included"], dtype=bool)
    if bool(np.all(included[mask] == include)):
        return
    included[mask] = include
    state["included"] = included
    _apply_inclusion_change(refresh_collapse=True)


def _on_point_click(trace, points, selector) -> None:
    """Click a marker to toggle that point's inclusion (grey = excluded)."""
    if _updating_from_code or not points.point_inds:
        return
    # customdata columns: [idx, T, a]
    cd = np.asarray(trace.customdata)
    local = int(points.point_inds[0])
    idx = int(cd[local, 0])
    included = np.asarray(state["included"], dtype=bool).copy()
    included[idx] = not included[idx]
    state["included"] = included
    _apply_inclusion_change(refresh_collapse=True)


def _on_box_selection(trace, points, selector) -> None:
    """Box/lasso select: toggle inclusion for all selected points on this trace."""
    if _updating_from_code or not points.point_inds:
        return
    cd = np.asarray(trace.customdata)
    idxs = [int(cd[i, 0]) for i in points.point_inds]
    included = np.asarray(state["included"], dtype=bool).copy()
    # If any selected point is included, exclude all; else include all (toggle group).
    if any(included[i] for i in idxs):
        for i in idxs:
            included[i] = False
    else:
        for i in idxs:
            included[i] = True
    state["included"] = included
    _apply_inclusion_change(refresh_collapse=True)


def _bind_collapse_interactions(fig: go.FigureWidget) -> None:
    L_values = state["L_values"]
    for i in range(len(L_values)):
        tr = fig.data[i]
        tr.on_click(_on_point_click)
        tr.on_selection(_on_box_selection)
        tr.on_change(_on_legend_visibility_change, "visible")


def _include_all_points(_btn=None) -> None:
    state["included"] = np.ones(len(state["df_full"]), dtype=bool)
    _apply_inclusion_change(refresh_collapse=True)


def _rebuild_figures_and_refresh() -> None:
    global collapse_fig, heatmap_fig, _updating_from_code
    _updating_from_code = True
    try:
        collapse_fig = _build_collapse_figure()
        heatmap_fig = _build_heatmap_figure()
        figures_box.children = (collapse_fig, heatmap_fig)
    finally:
        _updating_from_code = False
    params = _disc_params()
    _update_collapse(params)
    _update_heatmap(params)


def _sync_model_controls() -> None:
    if _is_additive_form():
        slider_sigma.description = "σ_g"
    else:
        slider_sigma.description = "σ_model"
    # t0,L0,p,q only enter a(t,L); unused for z-threshold gate.
    use_amp = state["disc_form"] != "additive_gp_z_threshold"
    for s in (slider_t0, slider_L0, slider_p, slider_q):
        s.disabled = not use_amp


def _on_slider_change(_change=None) -> None:
    params = _disc_params()
    status.value = "updating…"
    _update_collapse(params)
    _update_heatmap(params)


def _on_dataset_or_model_change(_change=None) -> None:
    family, name = dropdown_dataset.value
    form = str(dropdown_model.value)
    if (
        name == state.get("dataset")
        and family == state.get("family")
        and form == state.get("disc_form")
    ):
        return
    status.value = f"loading <b>{family}/{name}</b> / <b>{form}</b>…"
    try:
        load_dataset(name, family=family, discrepancy_form=form, quiet=True)
        _sync_model_controls()
        dataset_info.value = (
            f"<i>{_get_dataset(family, name).description}</i> "
            f"({family}/{name}: {len(state['df_full'])} points, "
            f"L={list(state['L_values'])})"
        )
        # Heatmap uses rebuilt tc_grid after family switch.
        _rebuild_figures_and_refresh()
    except Exception as exc:
        status.value = (
            f"<span style='color:#c44e52'>Failed to load {family}/{name}/{form}: {exc}</span>"
        )
        raise


slider_kwargs = dict(continuous_update=False, readout_format=".3g")
dropdown_dataset = widgets.Dropdown(
    options=AVAILABLE_DATASETS,
    value=(state["family"], state["dataset"]),
    description="dataset",
    layout=widgets.Layout(width="320px"),
    style={"description_width": "70px"},
)
dropdown_model = widgets.Dropdown(
    options=DISC_FORM_OPTIONS,
    value=state["disc_form"],
    description="model",
    layout=widgets.Layout(width="300px"),
    style={"description_width": "70px"},
)
dataset_info = widgets.HTML(
    value=(
        f"<i>{_get_dataset(state['family'], state['dataset']).description}</i> "
        f"({state['family']}/{state['dataset']}: {len(state['df_full'])} points, "
        f"L={list(state['L_values'])})"
    )
)

slider_t0 = widgets.FloatSlider(
    value=DISCREPANCY_T0_INIT,
    min=DISCREPANCY_T0_PRIOR_LOWER,
    max=DISCREPANCY_T0_PRIOR_UPPER,
    step=0.01,
    description="t0",
    **slider_kwargs,
)
slider_L0 = widgets.FloatSlider(
    value=DISCREPANCY_L0_INIT,
    min=DISCREPANCY_L0_PRIOR_LOWER,
    max=DISCREPANCY_L0_PRIOR_UPPER,
    step=1.0,
    description="L0",
    **slider_kwargs,
)
slider_p = widgets.FloatSlider(
    value=DISCREPANCY_P_INIT,
    min=DISCREPANCY_P_PRIOR_LOWER,
    max=DISCREPANCY_P_PRIOR_UPPER,
    step=0.05,
    description="p",
    **slider_kwargs,
)
slider_q = widgets.FloatSlider(
    value=DISCREPANCY_Q_INIT,
    min=DISCREPANCY_Q_PRIOR_LOWER,
    max=DISCREPANCY_Q_PRIOR_UPPER,
    step=0.05,
    description="q",
    **slider_kwargs,
)
slider_sigma = widgets.FloatSlider(
    value=DISCREPANCY_SIGMA_MODEL_INIT,
    min=0.0,
    max=SIGMA_SLIDER_MAX,
    step=0.05,
    description="σ_model",
    **slider_kwargs,
)
slider_K = widgets.FloatSlider(
    value=1.0,
    min=0.0,
    max=1.0,
    step=0.01,
    description="K",
    **slider_kwargs,
)
_sync_model_controls()

btn_include_all = widgets.Button(
    description="Include all points",
    tooltip="Reset inclusion mask to all points",
    layout=widgets.Layout(width="160px"),
)
btn_include_all.on_click(_include_all_points)

status = widgets.HTML(value="")
help_html = widgets.HTML(
    value=(
        "<b>Selection</b>: legend click toggles all points of that <code>L</code> "
        "(updates heatmap). Click a marker to include/exclude one point (grey = out). "
        "Box/lasso-drag to toggle a group. "
        "<b>Discrepancy</b>: noise / additive a·g use a=(|t|/t0)^p+(L0/L)^q. "
        "Z-threshold model: a=1 if |z|&gt;10 else 0 (t0,L0,p,q unused). "
        "Noise: σ_eff²=σ_MC²+(σ_model K a)². "
        "Additive: Φ=f+(… )g with amplitude σ_g (same slider). "
        "Try large <b>σ_g</b> to absorb gated points. "
        "<b>K</b> scales the correction (0=off, 1=full)."
    )
)

collapse_fig = _build_collapse_figure()
heatmap_fig = _build_heatmap_figure()
figures_box = widgets.HBox([collapse_fig, heatmap_fig])

for s in (slider_t0, slider_L0, slider_p, slider_q, slider_sigma, slider_K):
    s.observe(_on_slider_change, names="value")
dropdown_dataset.observe(_on_dataset_or_model_change, names="value")
dropdown_model.observe(_on_dataset_or_model_change, names="value")

controls = widgets.VBox(
    [
        widgets.HBox([dropdown_dataset, dropdown_model, btn_include_all]),
        dataset_info,
        help_html,
        widgets.HBox([slider_K, slider_sigma, slider_t0]),
        widgets.HBox([slider_L0, slider_p, slider_q]),
        status,
    ]
)
ui = widgets.VBox([controls, figures_box])
display(ui)

_update_collapse(_disc_params())
_update_heatmap(_disc_params())
